In [2]:
import re
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/plon/НедвижкаДанные/moscow_clean_base.csv")

In [3]:
def extract_area(title):
    match = re.search(r"\((\d+(?:[.,]\d+)?)\s*м²\)", str(title))
    if match:
        return float(match.group(1).replace(",", "."))
    return np.nan

def extract_rooms(title):
    match = re.match(r"^(\d+)-к кв\.", str(title))
    if match:
        return int(match.group(1))
    return np.nan

df["area"] = df["title"].apply(extract_area)
df["rooms"] = df["title"].apply(extract_rooms)

df["publication_time"] = pd.to_datetime(df["publication_time"], errors="coerce")
df["publication_year"] = df["publication_time"].dt.year
df["publication_month"] = df["publication_time"].dt.month

df["price_per_m2"] = df["price"] / df["area"]

In [4]:
df.head()

,request_year,request_region,id,url,title,price,source,city,city1,metro,address,person_type,publication_time,area,rooms,publication_year,publication_month,price_per_m2
0,2023,Москва,592510619,https://www.cian.ru/sale/flat/281955840,"3-к кв. Москва ул. Казакова, 7 (164.7 м²)",101479168,cian.ru,Москва,Москва,Курская,"ул. Казакова, 7",Агентство,2023-01-01 00:30:18,164.7,3.0,2023,1,616145.525197
1,2023,Москва,592512481,https://www.cian.ru/sale/flat/281956494,"1-к кв. Москва Волоколамское ш., 71к9 (35.0 м²)",10629570,cian.ru,Москва,Москва,Спартак,"Волоколамское ш., 71к9",Агентство,2023-01-01 00:30:19,35.0,1.0,2023,1,303702.000000
2,2023,Москва,592537336,https://www.cian.ru/sale/flat/281976963,"3-к кв. Москва Куркинское ш., 17 (96.8 м²)",24800000,cian.ru,Москва,Москва,СЗАО,"Куркинское ш., 17",Частное лицо (фильтр),2023-01-01 00:30:23,96.8,3.0,2023,1,256198.347107
3,2023,Москва,591008856,https://www.cian.ru/sale/commercial/280944356,"Торговая площадь в Москва ул. Остоженка, 1/9 (...",270000000,cian.ru,Москва,Москва,Кропоткинская,"ул. Остоженка, 1/9",Агентство,2023-01-01 00:49:37,492.0,NaN,2023,1,548780.487805
4,2023,Москва,591014495,https://www.cian.ru/sale/commercial/280948323,Помещение свободного назначения в Москва Огоро...,283493400,cian.ru,Москва,Москва,Бутырская,"Огородный проезд, 5",Агентство,2023-01-01 00:49:37,2054.3,NaN,2023,1,138000.000000


In [5]:
df["price_per_m2"].dtype

dtype('float64')

In [8]:
flats = df[df["rooms"].notna()].copy()

flats["rooms"] = flats["rooms"].astype(int)

Для дальнейшего анализа оставлены только квартиры, у которых удалось извлечь количество комнат из заголовка объявления.

Коммерческая недвижимость, парковки и другие типы объектов исключены, так как они имеют другую логику ценообразования и могут исказить модель оценки стоимости квартир.

In [10]:
flats.head()

,request_year,request_region,id,url,title,price,source,city,city1,metro,address,person_type,publication_time,area,rooms,publication_year,publication_month,price_per_m2
0,2023,Москва,592510619,https://www.cian.ru/sale/flat/281955840,"3-к кв. Москва ул. Казакова, 7 (164.7 м²)",101479168,cian.ru,Москва,Москва,Курская,"ул. Казакова, 7",Агентство,2023-01-01 00:30:18,164.7,3,2023,1,616145.525197
1,2023,Москва,592512481,https://www.cian.ru/sale/flat/281956494,"1-к кв. Москва Волоколамское ш., 71к9 (35.0 м²)",10629570,cian.ru,Москва,Москва,Спартак,"Волоколамское ш., 71к9",Агентство,2023-01-01 00:30:19,35.0,1,2023,1,303702.000000
2,2023,Москва,592537336,https://www.cian.ru/sale/flat/281976963,"3-к кв. Москва Куркинское ш., 17 (96.8 м²)",24800000,cian.ru,Москва,Москва,СЗАО,"Куркинское ш., 17",Частное лицо (фильтр),2023-01-01 00:30:23,96.8,3,2023,1,256198.347107
7,2023,Москва,592539705,https://www.cian.ru/sale/flat/281977924,"3-к кв. Москва Десеновское поселение, Новые Ва...",12675000,cian.ru,Москва,Москва,Ольховая,"Десеновское поселение, Новые Ватутинки централ...",Агентство,2023-01-01 02:29:56,67.6,3,2023,1,187500.000000
8,2023,Москва,592539761,https://www.cian.ru/sale/flat/281977822,"2-к кв. Москва Десеновское поселение, Новые Ва...",11533000,cian.ru,Москва,Москва,Ольховая,"Десеновское поселение, Новые Ватутинки централ...",Агентство,2023-01-01 02:29:57,60.7,2,2023,1,190000.000000


In [11]:
numeric_cols = ["price", "area", "price_per_m2"]

percentile_bounds = {}

for col in numeric_cols:
    lower_bound = flats[col].quantile(0.01)
    upper_bound = flats[col].quantile(0.99)
    percentile_bounds[col] = (lower_bound, upper_bound)

percentile_bounds

{'price': (np.float64(6507419.66), np.float64(262500000.0)),
 'area': (np.float64(27.0), np.float64(214.9)),
 'price_per_m2': (np.float64(155519.43383829427), np.float64(1600000.0))}

In [12]:
flats_filtered = flats.copy()

for col in numeric_cols:
    lower_bound, upper_bound = percentile_bounds[col]
    flats_filtered = flats_filtered[
        flats_filtered[col].between(lower_bound, upper_bound)
    ]

flats_filtered.shape

(1384363, 18)

In [13]:
removed_rows = len(flats) - len(flats_filtered)
removed_share = removed_rows / len(flats)

removed_rows, removed_share

(59312, 0.04108403899769685)

In [14]:
flats = flats_filtered.copy()

In [15]:
flats_filtered.to_csv("/Users/plon/НедвижкаДанные/moscow_flats_clean.csv", index=False)

## Обработка экстремальных значений

Для удаления выбросов использован процентильный подход.

По признакам `price`, `area` и `price_per_m2` были рассчитаны границы 1-го и 99-го процентилей. Наблюдения за пределами этих границ исключены из выборки.

Такой подход позволяет убрать экстремальные значения, которые могут быть ошибками парсинга, нестандартными объектами или редкими премиальными объявлениями, и при этом не задавать границы вручную.

После фильтрации датасет остается репрезентативным для массового рынка квартир Москвы.